In [1]:
import os, sys, shutil, multiprocessing, importlib.util

path = "/kaggle/input/datasets/pelmers/github-repository-metadata-with-5-stars/repo_metadata.json"

print("Python:", sys.version.split()[0])
print("File size (GB):", round(os.path.getsize(path) / 1e9, 3))
print("CPU cores:", multiprocessing.cpu_count())
with open("/proc/meminfo") as f:
    mem = {l.split(":")[0]: int(l.split()[1]) for l in f}
print("Total RAM (GB):", round(mem["MemTotal"] / 1e6, 1))
print("Available RAM (GB):", round(mem["MemAvailable"] / 1e6, 1))
print("Working disk free (GB):", round(shutil.disk_usage("/kaggle/working").free / 1e9, 1))
print("PySpark installed:", importlib.util.find_spec("pyspark") is not None)

# Peek at the file format without loading it
with open(path, "rb") as f:
    head = f.read(1500)
    f.seek(-300, os.SEEK_END)
    tail = f.read()
print("\n--- HEAD ---\n", head.decode("utf-8", errors="replace"))
print("\n--- TAIL ---\n", tail.decode("utf-8", errors="replace"))

Python: 3.13.15
File size (GB): 3.279
CPU cores: 4
Total RAM (GB): 32.9
Available RAM (GB): 31.7
Working disk free (GB): 20.9
PySpark installed: True

--- HEAD ---
 [{"owner": "freeCodeCamp", "name": "freeCodeCamp", "stars": 426893, "forks": 41377, "watchers": 8588, "isFork": false, "isArchived": false, "languages": [{"name": "TypeScript", "size": 2843131}, {"name": "JavaScript", "size": 725345}, {"name": "CSS", "size": 207745}, {"name": "Dockerfile", "size": 4446}, {"name": "HTML", "size": 1341}, {"name": "Shell", "size": 940}], "languageCount": 6, "topics": [{"name": "learn-to-code", "stars": 480}, {"name": "nonprofits", "stars": 44}, {"name": "programming", "stars": 1849}, {"name": "nodejs", "stars": 85145}, {"name": "react", "stars": 106415}, {"name": "d3", "stars": 76}, {"name": "careers", "stars": 60}, {"name": "education", "stars": 684}, {"name": "teachers", "stars": 39}, {"name": "javascript", "stars": 181110}], "topicCount": 16, "diskUsageKb": 512574, "pullRequests": 41267, "i

In [2]:
import json, os, time

src = "/kaggle/input/datasets/pelmers/github-repository-metadata-with-5-stars/repo_metadata.json"
out_dir = "/kaggle/temp/jsonl"
os.makedirs(out_dir, exist_ok=True)

SHARD_SIZE = 100_000
CHUNK = 64 * 1024 * 1024
dec = json.JSONDecoder()

t0 = time.time()
total, shard_id, in_shard = 0, 0, 0
out = open(f"{out_dir}/part-{shard_id:04d}.jsonl", "w", encoding="utf-8")

def emit(text):
    global total, shard_id, in_shard, out
    out.write(text + "\n")
    total += 1
    in_shard += 1
    if in_shard >= SHARD_SIZE:
        out.close()
        shard_id += 1
        in_shard = 0
        out = open(f"{out_dir}/part-{shard_id:04d}.jsonl", "w", encoding="utf-8")
        print(f"{total:,} records | {time.time()-t0:.0f}s", flush=True)

WS = " \n\r\t"
with open(src, "r", encoding="utf-8") as f:
    buf = f.read(CHUNK)
    pos = 0
    while pos < len(buf) and buf[pos] in WS:
        pos += 1
    assert buf[pos] == "[", "Expected a JSON array"
    pos += 1
    eof = False
    while True:
        while pos < len(buf) and buf[pos] in WS + ",":
            pos += 1
        if pos >= len(buf):
            if eof:
                break
            more = f.read(CHUNK)
            if not more:
                eof = True
            buf, pos = more, 0
            continue
        if buf[pos] == "]":
            break
        try:
            obj, end = dec.raw_decode(buf, pos)
        except json.JSONDecodeError:
            if eof:
                raise
            more = f.read(CHUNK)
            if not more:
                eof = True
            buf, pos = buf[pos:] + more, 0
            if len(buf) > 200 * 1024 * 1024:
                raise RuntimeError("Buffer too large; malformed object?")
            continue
        emit(buf[pos:end])
        pos = end

out.close()
last = f"{out_dir}/part-{shard_id:04d}.jsonl"
if os.path.getsize(last) == 0:
    os.remove(last)

files = sorted(os.listdir(out_dir))
size_gb = sum(os.path.getsize(f"{out_dir}/{x}") for x in files) / 1e9
print(f"\nDone: {total:,} records in {len(files)} shards, {size_gb:.2f} GB, {time.time()-t0:.0f}s")

100,000 records | 4s
200,000 records | 6s
300,000 records | 9s
400,000 records | 12s
500,000 records | 14s
600,000 records | 17s
700,000 records | 19s
800,000 records | 21s
900,000 records | 23s
1,000,000 records | 26s
1,100,000 records | 28s
1,200,000 records | 29s
1,300,000 records | 32s
1,400,000 records | 33s
1,500,000 records | 35s
1,600,000 records | 37s
1,700,000 records | 40s
1,800,000 records | 42s
1,900,000 records | 44s
2,000,000 records | 45s
2,100,000 records | 47s
2,200,000 records | 49s
2,300,000 records | 51s
2,400,000 records | 54s
2,500,000 records | 56s
2,600,000 records | 57s
2,700,000 records | 60s
2,800,000 records | 61s
2,900,000 records | 63s
3,000,000 records | 65s
3,100,000 records | 67s
3,200,000 records | 70s
3,300,000 records | 72s
3,400,000 records | 74s
3,500,000 records | 76s
3,600,000 records | 78s
3,700,000 records | 80s
3,800,000 records | 82s
3,900,000 records | 85s
4,000,000 records | 87s
4,100,000 records | 89s
4,200,000 records | 91s

Done: 4,247,

In [3]:
import time, os, shutil
from pyspark.sql import SparkSession

spark = (SparkSession.builder
    .master("local[4]")
    .appName("GitHubRepoMetadata")
    .config("spark.driver.memory", "20g")
    .config("spark.sql.shuffle.partitions", "16")
    .config("spark.local.dir", "/kaggle/temp/spark")
    .getOrCreate())
print("Spark version:", spark.version)

# 1. Read JSON Lines shards (schema inference = one full pass)
t0 = time.time()
df = spark.read.json("/kaggle/temp/jsonl/*.jsonl")
print(f"Schema inferred in {time.time()-t0:.0f}s")
df.printSchema()

# 2. Write Parquet
pq = "/kaggle/working/parquet/repos"
shutil.rmtree(pq, ignore_errors=True)
t0 = time.time()
df.write.mode("overwrite").parquet(pq)
print(f"Parquet written in {time.time()-t0:.0f}s")
size = sum(os.path.getsize(os.path.join(d, f)) for d, _, fs in os.walk(pq) for f in fs)
print(f"Parquet size: {size/1e9:.2f} GB")

# 3. Compare: count from raw JSON vs Parquet
t0 = time.time(); n_json = spark.read.json("/kaggle/temp/jsonl/*.jsonl").count()
print(f"JSON count: {n_json:,} in {time.time()-t0:.0f}s")
t0 = time.time(); n_pq = spark.read.parquet(pq).count()
print(f"Parquet count: {n_pq:,} in {time.time()-t0:.1f}s")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/29 19:45:03 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/29 19:45:03 WARN SparkConf: Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in standalone/kubernetes and LOCAL_DIRS in YARN).


Spark version: 4.0.4


26/10/29 19:45:11 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: /kaggle/temp/jsonl/*.jsonl.
java.io.FileNotFoundException: File /kaggle/temp/jsonl/*.jsonl does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:917)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1238)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:907)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.FileStreamSink$.hasMetadata(FileStreamSink.scala:56)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:381)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:143)
	at org.apache.spark.sql.catalyst.analysis

Schema inferred in 35s
root
 |-- assignableUserCount: long (nullable = true)
 |-- codeOfConduct: string (nullable = true)
 |-- createdAt: string (nullable = true)
 |-- defaultBranchCommitCount: long (nullable = true)
 |-- description: string (nullable = true)
 |-- diskUsageKb: long (nullable = true)
 |-- forkingAllowed: boolean (nullable = true)
 |-- forks: long (nullable = true)
 |-- isArchived: boolean (nullable = true)
 |-- isFork: boolean (nullable = true)
 |-- issues: long (nullable = true)
 |-- languageCount: long (nullable = true)
 |-- languages: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- name: string (nullable = true)
 |    |    |-- size: long (nullable = true)
 |-- license: string (nullable = true)
 |-- name: string (nullable = true)
 |-- nameWithOwner: string (nullable = true)
 |-- owner: string (nullable = true)
 |-- parent: string (nullable = true)
 |-- primaryLanguage: string (nullable = true)
 |-- pullRequests: long (nullable = 

26/10/29 19:46:36 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: /kaggle/temp/jsonl/*.jsonl.
java.io.FileNotFoundException: File /kaggle/temp/jsonl/*.jsonl does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:917)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1238)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:907)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.FileStreamSink$.hasMetadata(FileStreamSink.scala:56)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:381)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:143)
	at org.apache.spark.sql.catalyst.analysis

Parquet written in 51s
Parquet size: 0.66 GB


JSON count: 4,247,611 in 42s
Parquet count: 4,247,611 in 0.7s


In [4]:
import time
import pandas as pd
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder.master("local[4]").appName("GitHubRepoMetadata")
         .config("spark.driver.memory", "20g")
         .config("spark.sql.shuffle.partitions", "16").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

df = spark.read.parquet("/kaggle/working/parquet/repos")
t0 = time.time()

# 1. Null counts per column
nulls = df.select([F.count(F.when(F.col(c).isNull(), 1)).alias(c) for c in df.columns]).first().asDict()
print("NULL COUNTS")
print(pd.Series(nulls).to_string())

# 2. Duplicates and key sanity
total = df.count()
distinct_keys = df.select("nameWithOwner").distinct().count()
print(f"\nTotal rows: {total:,} | distinct nameWithOwner: {distinct_keys:,} | duplicates: {total - distinct_keys:,}")

# 3. Numeric ranges and percentiles
num_cols = ["stars", "forks", "watchers", "issues", "pullRequests", "diskUsageKb",
            "defaultBranchCommitCount", "assignableUserCount", "languageCount", "topicCount"]
aggs = []
for c in num_cols:
    aggs += [F.min(c).alias(f"{c}__min"),
             F.percentile_approx(c, 0.5).alias(f"{c}__median"),
             F.percentile_approx(c, 0.99).alias(f"{c}__p99"),
             F.max(c).alias(f"{c}__max")]
r = df.agg(*aggs).first().asDict()
summary = pd.DataFrame({c: {k: r[f"{c}__{k}"] for k in ["min", "median", "p99", "max"]} for c in num_cols}).T
print("\nNUMERIC SUMMARY")
print(summary.to_string())

# 4. Dates and flags
d = df.select(F.to_timestamp("createdAt").alias("c"), F.to_timestamp("pushedAt").alias("p"),
              "isFork", "isArchived", "parent")
dr = d.agg(F.min("c").alias("created_min"), F.max("c").alias("created_max"),
           F.min("p").alias("pushed_min"), F.max("p").alias("pushed_max"),
           F.sum((F.col("p") < F.col("c")).cast("int")).alias("pushed_before_created"),
           F.sum(F.col("c").isNull().cast("int")).alias("created_unparsed_or_null"),
           F.sum(F.col("isFork").cast("int")).alias("forks_flagged"),
           F.sum(F.col("isArchived").cast("int")).alias("archived"),
           F.sum((F.col("isFork") & F.col("parent").isNull()).cast("int")).alias("fork_without_parent"),
           F.sum((~F.col("isFork") & F.col("parent").isNotNull()).cast("int")).alias("parent_but_not_fork"),
          ).first().asDict()
print("\nDATES / FLAGS")
for k, v in dr.items():
    print(f"{k}: {v}")

print(f"\nProfiling took {time.time()-t0:.0f}s")

NULL COUNTS
assignableUserCount               0
codeOfConduct               4060923
createdAt                         0
defaultBranchCommitCount      14807
description                  641869
diskUsageKb                       0
forkingAllowed                    0
forks                             0
isArchived                        0
isFork                            0
issues                            0
languageCount                     0
languages                         0
license                     2038367
name                              0
nameWithOwner                     0
owner                             0
parent                      4247611
primaryLanguage              365289
pullRequests                      0
pushedAt                        122
stars                             0
topicCount                        0
topics                            0
watchers                          0



Total rows: 4,247,611 | distinct nameWithOwner: 4,242,532 | duplicates: 5,079



NUMERIC SUMMARY
                          min  median     p99        max
stars                       4      12    1342     426893
forks                       0       3     284     243520
watchers                    0       2      64       9879
issues                      0       1     261     420642
pullRequests                0       0     501     388408
diskUsageKb                 0     609  481149  115580150
defaultBranchCommitCount    1      25    3048   12602434
assignableUserCount         0       1      32       9996
languageCount               0       2      11        489
topicCount                  0       0      16         20



DATES / FLAGS
created_min: 2009-01-01 00:38:50
created_max: 2025-09-01 23:50:33
pushed_min: 2008-05-05 01:53:29
pushed_max: 2025-09-13 12:33:13
pushed_before_created: 17872
created_unparsed_or_null: 0
forks_flagged: 0
archived: 256301
fork_without_parent: 0
parent_but_not_fork: 0

Profiling took 45s


In [5]:
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder.master("local[4]").appName("GitHubRepoMetadata")
         .config("spark.driver.memory", "20g")
         .config("spark.sql.shuffle.partitions", "16").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

df = spark.read.parquet("/kaggle/working/parquet/repos")
key = "nameWithOwner"
scalar = [c for c in df.columns if c not in ("languages", "topics", key)]

dup_keys = df.groupBy(key).count().filter("count > 1").cache()
print("Keys appearing more than once:", dup_keys.count())
print("Copies-per-key distribution:")
dup_keys.groupBy("count").count().orderBy("count").show()

dup_rows = df.join(F.broadcast(dup_keys.select(key)), key, "inner")
per_key = dup_rows.groupBy(key).agg(
    F.count("*").alias("n_rows"),
    F.countDistinct(F.to_json(F.struct(*df.columns))).alias("distinct_versions"),
    *[F.countDistinct(c).alias(c) for c in scalar]
).cache()

print("Identical copies (1 distinct version):", per_key.filter("distinct_versions = 1").count())
print("Differing copies (>1 version):        ", per_key.filter("distinct_versions > 1").count())

print("\nNumber of duplicate keys where each column differs:")
diff = per_key.select([F.sum((F.col(c) > 1).cast("int")).alias(c) for c in scalar]).first().asDict()
for c, v in sorted(diff.items(), key=lambda kv: -kv[1]):
    if v: print(f"  {c}: {v}")

print("\nExamples:")
(dup_rows.orderBy(key, "pushedAt")
    .select(key, "stars", "forks", "createdAt", "pushedAt", "defaultBranchCommitCount")
    .limit(12).show(truncate=False))

lower_distinct = df.select(F.lower(F.col(key))).distinct().count()
print("Distinct nameWithOwner, case-insensitive:", f"{lower_distinct:,}")

Keys appearing more than once: 5033
Copies-per-key distribution:
+-----+-----+
|count|count|
+-----+-----+
|    2| 4992|
|    3|   38|
|    4|    2|
|    6|    1|
+-----+-----+



Identical copies (1 distinct version): 78
Differing copies (>1 version):         4955

Number of duplicate keys where each column differs:
  stars: 4953
  pushedAt: 264
  forks: 256
  diskUsageKb: 241
  defaultBranchCommitCount: 195
  pullRequests: 100
  watchers: 85
  issues: 79
  assignableUserCount: 20
  topicCount: 5
  description: 3
  languageCount: 3
  createdAt: 1
  license: 1
  primaryLanguage: 1

Examples:


+---------------------------+-----+-----+--------------------+--------------------+------------------------+
|nameWithOwner              |stars|forks|createdAt           |pushedAt            |defaultBranchCommitCount|
+---------------------------+-----+-----+--------------------+--------------------+------------------------+
|029danio/clash             |175  |18   |2024-11-30T07:29:55Z|2025-02-27T10:20:36Z|27                      |
|029danio/clash             |173  |18   |2024-11-30T07:29:55Z|2025-02-27T10:20:36Z|27                      |
|0PeterAdel/CyberBrain_Model|17   |1    |2025-02-26T08:14:25Z|2025-02-26T11:49:34Z|16                      |
|0PeterAdel/CyberBrain_Model|16   |1    |2025-02-26T08:14:25Z|2025-02-26T11:49:34Z|16                      |
|0eix/roll-the-ball         |5    |2    |2019-07-26T12:29:07Z|2020-01-19T23:42:56Z|3                       |
|0eix/roll-the-ball         |6    |2    |2019-07-26T12:29:07Z|2020-01-19T23:42:56Z|3                       |
|0joseDark/gravador

Distinct nameWithOwner, case-insensitive: 4,242,532


In [6]:
import time
from pyspark.sql import SparkSession, Window, functions as F

spark = (SparkSession.builder.master("local[4]").appName("GitHubRepoMetadata")
         .config("spark.driver.memory", "20g")
         .config("spark.sql.shuffle.partitions", "16").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.session.timeZone", "UTC")

src = "/kaggle/working/parquet/repos"
dst = "/kaggle/working/parquet/clean"
t0 = time.time()

raw = spark.read.parquet(src)
n_raw = raw.count()

# 1. Parse dates
df = (raw.withColumn("created_ts", F.to_timestamp("createdAt"))
         .withColumn("pushed_ts", F.to_timestamp("pushedAt")))

# 2. Deduplicate: latest pushedAt, then most stars, then most forks
w = Window.partitionBy("nameWithOwner").orderBy(
    F.col("pushed_ts").desc_nulls_last(), F.col("stars").desc(), F.col("forks").desc())
df = df.withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn")

# 3. Snapshot date = latest pushedAt in the data (as epoch seconds, avoids timezone issues)
snap_epoch = df.agg(F.max(F.unix_timestamp("pushed_ts"))).first()[0]
print("Snapshot (max pushedAt, UTC):", spark.range(1).select(F.from_unixtime(F.lit(snap_epoch))).first()[0])

DAY = 86400.0
valid_push = F.col("pushed_ts").isNotNull() & (F.col("pushed_ts") >= F.col("created_ts"))

clean = (df
    .withColumn("pushed_missing", F.col("pushed_ts").isNull())
    .withColumn("date_anomaly", F.coalesce(F.col("pushed_ts") < F.col("created_ts"), F.lit(False)))
    .withColumn("created_year", F.year("created_ts"))
    .withColumn("age_days", (F.lit(snap_epoch) - F.unix_timestamp("created_ts")) / DAY)
    .withColumn("days_since_push",
                F.when(valid_push, (F.lit(snap_epoch) - F.unix_timestamp("pushed_ts")) / DAY))
    .withColumn("active_span_days",
                F.when(valid_push, (F.unix_timestamp("pushed_ts") - F.unix_timestamp("created_ts")) / DAY))
    .withColumn("has_license", F.col("license").isNotNull())
    .withColumn("license_clean", F.coalesce("license", F.lit("No license")))
    .withColumn("primary_language_clean", F.coalesce("primaryLanguage", F.lit("Unknown")))
    .withColumn("has_description", F.coalesce(F.length(F.trim("description")) > 0, F.lit(False)))
    .withColumn("has_code_of_conduct", F.col("codeOfConduct").isNotNull())
    .withColumn("has_topics", F.col("topicCount") > 0)
    .withColumn("commit_count_missing", F.col("defaultBranchCommitCount").isNull())
    .withColumn("total_language_bytes",
                F.aggregate("languages", F.lit(0).cast("long"),
                            lambda acc, x: acc + F.coalesce(x["size"], F.lit(0).cast("long"))))
    .drop("isFork", "parent", "createdAt", "pushedAt"))

clean.write.mode("overwrite").parquet(dst)
print(f"Written in {time.time()-t0:.0f}s")

# 4. Validate by reading back
chk = spark.read.parquet(dst)
n_clean = chk.count()
n_keys = chk.select("nameWithOwner").distinct().count()
print(f"\nRaw rows: {n_raw:,} | clean rows: {n_clean:,} | removed: {n_raw-n_clean:,}")
print(f"Distinct keys: {n_keys:,} (should equal clean rows)")

r = chk.agg(
    F.sum(F.col("pushed_missing").cast("int")).alias("pushed_missing"),
    F.sum(F.col("date_anomaly").cast("int")).alias("date_anomaly"),
    F.sum(F.col("commit_count_missing").cast("int")).alias("commit_count_missing"),
    F.sum(F.col("has_license").cast("int")).alias("has_license"),
    F.sum(F.col("has_description").cast("int")).alias("has_description"),
    F.sum(F.col("has_topics").cast("int")).alias("has_topics"),
    F.count(F.when(F.col("days_since_push").isNull(), 1)).alias("days_since_push_null"),
    F.percentile_approx("age_days", [0.0, 0.5, 1.0]).alias("age_days_min_med_max"),
    F.percentile_approx("days_since_push", [0.0, 0.5, 1.0]).alias("days_since_push_min_med_max"),
    F.min("stars").alias("stars_min"),
).first().asDict()
for k, v in r.items():
    print(f"{k}: {v}")
chk.printSchema()

Snapshot (max pushedAt, UTC): 2025-09-13 12:33:13


Written in 69s



Raw rows: 4,247,611 | clean rows: 4,242,532 | removed: 5,079
Distinct keys: 4,242,532 (should equal clean rows)


pushed_missing: 122
date_anomaly: 17837
commit_count_missing: 14801
has_license: 2206358
has_description: 3601431
has_topics: 1282954
days_since_push_null: 17959
age_days_min_med_max: [11.52962962962963, 2140.131712962963, 6099.4960995370375]
days_since_push_min_med_max: [0.0, 1111.8504513888888, 6098.9878240740745]
stars_min: 4
root
 |-- assignableUserCount: long (nullable = true)
 |-- codeOfConduct: string (nullable = true)
 |-- defaultBranchCommitCount: long (nullable = true)
 |-- description: string (nullable = true)
 |-- diskUsageKb: long (nullable = true)
 |-- forkingAllowed: boolean (nullable = true)
 |-- forks: long (nullable = true)
 |-- isArchived: boolean (nullable = true)
 |-- issues: long (nullable = true)
 |-- languageCount: long (nullable = true)
 |-- languages: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- name: string (nullable = true)
 |    |    |-- size: long (nullable = true)
 |-- license: string (nullable = true)
 |-- name: 

In [7]:
import os, time
import pandas as pd
from pyspark.sql import SparkSession, functions as F
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.stat import Correlation

spark = (SparkSession.builder.master("local[4]").appName("GitHubRepoMetadata")
         .config("spark.driver.memory", "20g")
         .config("spark.sql.shuffle.partitions", "16").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

OUT = "/kaggle/working/outputs"
os.makedirs(OUT, exist_ok=True)
timings = {}

def save(name, data):
    t = time.time()
    pdf = data if isinstance(data, pd.DataFrame) else data.toPandas()
    pdf.to_csv(f"{OUT}/{name}.csv", index=False)
    timings[name] = (len(pdf), round(time.time() - t, 1))
    return pdf

def med(c, alias=None):
    return F.percentile_approx(c, 0.5).alias(alias or f"median_{c}")

def share(cond, alias):
    return F.avg(cond.cast("int")).alias(alias)   # avg ignores nulls

def bucketize(col, edges, labels):
    """edges = inclusive upper bounds for all but the last label. Column must be non-null."""
    order, label = F.lit(len(labels)), F.lit(labels[-1])
    for i in range(len(edges) - 1, -1, -1):
        cond = F.col(col) <= edges[i]
        order = F.when(cond, i + 1).otherwise(order)
        label = F.when(cond, labels[i]).otherwise(label)
    return order, label

def bucket_table(name, src, col, edges, labels):
    order, label = bucketize(col, edges, labels)
    t = (src.select(order.alias("bucket_order"), label.alias("bucket"),
                    "stars", "forks", "pullRequests", "defaultBranchCommitCount")
            .groupBy("bucket_order", "bucket")
            .agg(F.count("*").alias("repos"), med("stars"), med("forks"),
                 med("defaultBranchCommitCount", "median_commits"),
                 share(F.col("pullRequests") > 0, "share_with_pull_requests"))
            .orderBy("bucket_order"))
    return save(name, t)

df = spark.read.parquet("/kaggle/working/parquet/clean")

core = (df.select("stars", "forks", "watchers", "issues", "pullRequests", "diskUsageKb",
                  "defaultBranchCommitCount", "assignableUserCount", "isArchived",
                  "primary_language_clean", "license_clean", "has_license", "has_description",
                  "has_code_of_conduct", "has_topics", "created_year", "age_days", "days_since_push")
        .withColumn("is_active_365d",
                    F.when(F.col("days_since_push").isNotNull(), F.col("days_since_push") <= 365))
        .withColumn("log10_stars_bin", F.floor(F.log10("stars") * 4) / 4)
        .cache())
t0 = time.time()
n = core.count()
print(f"Cached {n:,} rows in {time.time()-t0:.0f}s")
T_ALL = time.time()

# 1. Star percentiles and overview
qs = [0.5, 0.75, 0.9, 0.95, 0.99, 0.999]
pct = core.agg(F.percentile_approx("stars", qs, 10000).alias("p")).first()["p"]
stars_pct = save("stars_percentiles", pd.DataFrame({"percentile": qs, "stars": pct}))

ov = core.agg(
    F.count("*").alias("repos"), F.sum("stars").alias("total_stars"),
    share(F.col("isArchived"), "share_archived"),
    share(F.col("has_license"), "share_licensed"),
    share(F.col("has_description"), "share_with_description"),
    share(F.col("has_topics"), "share_with_topics"),
    share(F.col("has_code_of_conduct"), "share_with_code_of_conduct"),
    share(F.col("is_active_365d"), "share_active_365d_of_valid_dates"),
    share(F.col("pullRequests") > 0, "share_with_pull_requests"),
    med("age_days"), med("days_since_push")).toPandas()
top = core.filter(F.col("stars") >= pct[4]).agg(F.count("*"), F.sum("stars")).first()
ov["p99_stars_threshold"] = pct[4]
ov["repos_at_or_above_p99"] = top[0]
ov["share_of_repos_at_or_above_p99"] = top[0] / n
ov["share_of_all_stars_at_or_above_p99"] = top[1] / ov.loc[0, "total_stars"]
ov = save("overview", ov)

# 2. Star distribution (log10 bins of width 0.25)
save("stars_log_hist", core.groupBy(F.col("log10_stars_bin").alias("log10_stars_lower"))
     .count().orderBy("log10_stars_lower"))

# 3. Primary language summary (top 100 by repo count)
lang = save("language_summary", core.groupBy("primary_language_clean").agg(
    F.count("*").alias("repos"), F.sum("stars").alias("total_stars"), med("stars"),
    F.avg("forks").alias("avg_forks"),
    share(F.col("isArchived"), "share_archived"),
    share(F.col("is_active_365d"), "share_active_365d")
).orderBy(F.desc("repos")).limit(100))

# 4. Language usage across ALL languages in each repo (bytes)
save("language_bytes", df.select(F.explode("languages").alias("l"))
     .select(F.col("l.name").alias("language"), F.col("l.size").alias("bytes"))
     .groupBy("language").agg(F.count("*").alias("repos_using"), F.sum("bytes").alias("total_bytes"))
     .orderBy(F.desc("repos_using")).limit(100))

# 5. Creation-year cohorts
cohort = save("created_year_cohorts", core.groupBy("created_year").agg(
    F.count("*").alias("repos"), med("stars"), med("forks"),
    share(F.col("isArchived"), "share_archived"),
    share(F.col("is_active_365d"), "share_active_365d")).orderBy("created_year"))

# 6. Archived vs active
aa = save("archived_vs_active", core.groupBy("isArchived").agg(
    F.count("*").alias("repos"), med("stars"), med("forks"), med("watchers"),
    med("pullRequests"), med("issues"), med("defaultBranchCommitCount", "median_commits"),
    med("age_days"), med("days_since_push")))

# 7. Licenses
save("license_summary", core.groupBy("license_clean").agg(
    F.count("*").alias("repos"), med("stars"), med("forks")).orderBy(F.desc("repos")).limit(25))
save("license_flag", core.groupBy("has_license").agg(
    F.count("*").alias("repos"), med("stars"), med("forks"), med("pullRequests")))

# 8. Collaboration, size, recency buckets
bucket_table("collab_buckets", core, "assignableUserCount", [1, 5, 20], ["0-1", "2-5", "6-20", "21+"])
bucket_table("size_buckets", core, "diskUsageKb", [100, 1024, 10240, 102400],
             ["<=100 KB", "100 KB-1 MB", "1-10 MB", "10-100 MB", ">100 MB"])
bucket_table("recency_buckets", core.filter(F.col("days_since_push").isNotNull()), "days_since_push",
             [30, 90, 365, 730, 1825], ["<=30d", "31-90d", "91-365d", "1-2y", "2-5y", ">5y"])

# 9. Topics (top 100)
save("topic_summary", df.select("stars", F.explode("topics").alias("t"))
     .select(F.col("t.name").alias("topic"), "stars")
     .groupBy("topic").agg(F.count("*").alias("repos"), med("stars"), F.sum("stars").alias("total_stars"))
     .orderBy(F.desc("repos")).limit(100))

# 10. Correlation (Pearson on log1p because the data is heavily skewed)
ccols = ["stars", "forks", "watchers", "issues", "pullRequests", "diskUsageKb",
         "defaultBranchCommitCount", "assignableUserCount", "age_days"]
logd = core.dropna(subset=ccols).select([F.log1p(c).alias(c) for c in ccols])
vec = VectorAssembler(inputCols=ccols, outputCol="v").transform(logd).select("v")
m = Correlation.corr(vec, "v", "pearson").head()[0].toArray()
corr = pd.DataFrame(m, index=ccols, columns=ccols).round(4)
save("correlation_log1p", corr.rename_axis("variable").reset_index())

# ---- Report ----
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)
print(f"\nAll tables computed in {time.time()-T_ALL:.0f}s (after {n:,}-row cache)\n")
print("OVERVIEW");            print(ov.T.to_string())
print("\nSTAR PERCENTILES");  print(stars_pct.to_string(index=False))
print("\nARCHIVED vs ACTIVE"); print(aa.set_index("isArchived").T.to_string())
print("\nTOP 10 PRIMARY LANGUAGES");
print(lang.head(10)[["primary_language_clean", "repos", "median_stars", "share_archived"]].to_string(index=False))
print("\nCREATED-YEAR COHORTS"); print(cohort.to_string(index=False))
print("\nCORRELATION (Pearson on log1p)"); print(corr.round(2).to_string())
print("\nOUTPUT FILES (rows, seconds)")
for k, v in timings.items(): print(f"  {k}.csv: {v[0]} rows, {v[1]}s")

Cached 4,242,532 rows in 11s



All tables computed in 77s (after 4,242,532-row cache)

OVERVIEW
                                               0
repos                               4.242532e+06
total_stars                         4.129261e+08
share_archived                      6.038611e-02
share_licensed                      5.200569e-01
share_with_description              8.488872e-01
share_with_topics                   3.024029e-01
share_with_code_of_conduct          4.394098e-02
share_active_365d_of_valid_dates    2.363323e-01
share_with_pull_requests            4.879520e-01
median_age_days                     2.140132e+03
median_days_since_push              1.111850e+03
p99_stars_threshold                 1.333000e+03
repos_at_or_above_p99               4.250800e+04
share_of_repos_at_or_above_p99      1.001949e-02
share_of_all_stars_at_or_above_p99  5.525912e-01

STAR PERCENTILES
 percentile  stars
      0.500     12
      0.750     30
      0.900     97
      0.950    228
      0.990   1333
      0.999  10294

In [8]:
import time, os
import pandas as pd
from pyspark.sql import SparkSession, functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression, GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator

spark = (SparkSession.builder.master("local[4]").appName("GitHubRepoMetadata")
         .config("spark.driver.memory", "20g")
         .config("spark.sql.shuffle.partitions", "16").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

OUT = "/kaggle/working/outputs"
os.makedirs(OUT, exist_ok=True)
THRESH = 100   # "popular" = at least 100 stars

df = spark.read.parquet("/kaggle/working/parquet/clean")
TOP_LANGS = [r[0] for r in df.groupBy("primary_language_clean").count()
             .orderBy(F.desc("count")).limit(20).collect()]

ml = (df.select(
        "nameWithOwner",
        (F.col("stars") >= THRESH).cast("double").alias("label"),
        F.log1p("forks").alias("log_forks"),
        F.log1p("watchers").alias("log_watchers"),
        F.log1p("issues").alias("log_issues"),
        F.log1p("pullRequests").alias("log_pullRequests"),
        F.log1p("diskUsageKb").alias("log_diskUsageKb"),
        F.log1p(F.coalesce("defaultBranchCommitCount", F.lit(0))).alias("log_commits"),
        F.log1p("assignableUserCount").alias("log_assignableUsers"),
        F.log1p("languageCount").alias("log_languageCount"),
        F.log1p("topicCount").alias("log_topicCount"),
        F.log1p("age_days").alias("log_age_days"),
        F.log1p(F.coalesce("days_since_push", F.lit(0.0))).alias("log_days_since_push"),
        (F.col("pushed_missing") | F.col("date_anomaly")).cast("double").alias("date_invalid"),
        F.col("commit_count_missing").cast("double").alias("commit_count_missing"),
        F.col("isArchived").cast("double").alias("is_archived"),
        F.col("has_license").cast("double").alias("has_license"),
        F.col("has_description").cast("double").alias("has_description"),
        F.col("has_code_of_conduct").cast("double").alias("has_code_of_conduct"),
        F.col("has_topics").cast("double").alias("has_topics"),
        F.when(F.col("primary_language_clean").isin(TOP_LANGS),
               F.col("primary_language_clean")).otherwise("Other").alias("lang"))
      .withColumn("split", F.pmod(F.hash("nameWithOwner"), F.lit(10)))
      .cache())

train = ml.filter("split < 8")
test = ml.filter("split >= 8").cache()
n_tr, n_te = train.count(), test.count()
p_tr = train.agg(F.avg("label")).first()[0]
p_te = test.agg(F.avg("label")).first()[0]
print(f"Train {n_tr:,} (popular share {p_tr:.4f}) | Test {n_te:,} (popular share {p_te:.4f})")
print(f"Baseline PR-AUC for a random classifier = popular share = {p_te:.4f}")
train = train.withColumn("w", F.when(F.col("label") == 1.0, (1 - p_tr) / p_tr).otherwise(1.0))

BASE = ["log_issues", "log_pullRequests", "log_diskUsageKb", "log_commits",
        "log_assignableUsers", "log_languageCount", "log_topicCount", "log_age_days",
        "log_days_since_push", "date_invalid", "commit_count_missing", "is_archived",
        "has_license", "has_description", "has_code_of_conduct", "has_topics"]
ENGAGE = ["log_forks", "log_watchers"]

def evaluate(pred):
    cm = pred.groupBy("label", "prediction").count().toPandas()
    g = lambda l, p: int(cm[(cm.label == l) & (cm.prediction == p)]["count"].sum())
    tp, fp, fn, tn = g(1.0, 1.0), g(0.0, 1.0), g(1.0, 0.0), g(0.0, 0.0)
    prec = tp / (tp + fp) if tp + fp else 0.0
    rec = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
    pr = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction",
                                       metricName="areaUnderPR").evaluate(pred)
    roc = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction",
                                        metricName="areaUnderROC").evaluate(pred)
    return dict(TP=tp, FP=fp, FN=fn, TN=tn, precision=prec, recall=rec, f1=f1,
                accuracy=(tp + tn) / (tp + fp + fn + tn), pr_auc=pr, roc_auc=roc)

results, importances = [], []
for set_name, extra in [("A_no_forks_watchers", []), ("B_with_forks_watchers", ENGAGE)]:
    num = BASE + extra
    for model_name in ["LogisticRegression", "GBT"]:
        if model_name == "LogisticRegression":
            clf = LogisticRegression(featuresCol="features", labelCol="label", weightCol="w",
                                     maxIter=50, regParam=1e-4)
        else:
            clf = GBTClassifier(featuresCol="features", labelCol="label", weightCol="w",
                                maxIter=40, maxDepth=5, maxBins=64, seed=42)
        pipe = Pipeline(stages=[
            StringIndexer(inputCol="lang", outputCol="lang_idx", handleInvalid="keep"),
            OneHotEncoder(inputCol="lang_idx", outputCol="lang_ohe", dropLast=False),
            VectorAssembler(inputCols=num + ["lang_ohe"], outputCol="features"),
            clf])
        t0 = time.time()
        model = pipe.fit(train)
        t_fit = time.time() - t0
        pred = model.transform(test).select("label", "prediction", "rawPrediction")
        m = evaluate(pred)
        m.update(feature_set=set_name, model=model_name, fit_seconds=round(t_fit, 1))
        results.append(m)
        print(f"{set_name} | {model_name}: fit {t_fit:.0f}s | P={m['precision']:.3f} "
              f"R={m['recall']:.3f} F1={m['f1']:.3f} PR-AUC={m['pr_auc']:.3f} ROC-AUC={m['roc_auc']:.3f}",
              flush=True)
        if model_name == "GBT":
            names = num + [f"lang={l}" for l in model.stages[0].labels] + ["lang=_unseen"]
            imp = model.stages[-1].featureImportances.toArray()
            importances += [dict(feature_set=set_name, feature=n, importance=float(v))
                            for n, v in zip(names, imp)]

res = pd.DataFrame(results)
cols = ["feature_set", "model", "TP", "FP", "FN", "TN", "precision", "recall", "f1",
        "accuracy", "pr_auc", "roc_auc", "fit_seconds"]
res = res[cols]
res.to_csv(f"{OUT}/ml_results.csv", index=False)
imp_df = pd.DataFrame(importances).sort_values(["feature_set", "importance"], ascending=[True, False])
imp_df.to_csv(f"{OUT}/ml_feature_importance_gbt.csv", index=False)

pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)
print("\nRESULTS (test set, popular class = stars >= %d)" % THRESH)
print(res.round(3).to_string(index=False))
print("\nTOP GBT FEATURES")
for s, g in imp_df.groupby("feature_set"):
    print(s); print(g.head(8).round(3).to_string(index=False))

Train 3,395,758 (popular share 0.0983) | Test 846,774 (popular share 0.0981)
Baseline PR-AUC for a random classifier = popular share = 0.0981


A_no_forks_watchers | LogisticRegression: fit 31s | P=0.313 R=0.753 F1=0.442 PR-AUC=0.476 ROC-AUC=0.862


A_no_forks_watchers | GBT: fit 214s | P=0.304 R=0.811 F1=0.442 PR-AUC=0.553 ROC-AUC=0.885


B_with_forks_watchers | LogisticRegression: fit 27s | P=0.512 R=0.915 F1=0.657 PR-AUC=0.826 ROC-AUC=0.969


B_with_forks_watchers | GBT: fit 232s | P=0.503 R=0.923 F1=0.651 PR-AUC=0.840 ROC-AUC=0.971

RESULTS (test set, popular class = stars >= 100)
          feature_set              model    TP     FP    FN     TN  precision  recall    f1  accuracy  pr_auc  roc_auc  fit_seconds
  A_no_forks_watchers LogisticRegression 62518 137373 20513 626370      0.313   0.753 0.442     0.814   0.476    0.862         30.7
  A_no_forks_watchers                GBT 67321 154027 15710 609716      0.304   0.811 0.442     0.800   0.553    0.885        214.0
B_with_forks_watchers LogisticRegression 75982  72348  7049 691395      0.512   0.915 0.657     0.906   0.826    0.969         26.5
B_with_forks_watchers                GBT 76604  75587  6427 688156      0.503   0.923 0.651     0.903   0.840    0.971        232.3

TOP GBT FEATURES
A_no_forks_watchers
        feature_set             feature  importance
A_no_forks_watchers          log_issues       0.421
A_no_forks_watchers    log_pullRequests       0.138
A_no